# Seaborn 기초 - 분포와 범주 시각화

## 문제 상황
숫자 비교엔 약하고 모양·색 차이엔 강한 사람 눈을 위한 도구
- Matplotlib은 강력하지만 손이 많이 가는 도구 - Seaborn은 적은 코드로 통계 그래프를 그려 주는
  전문 공구함 - Pandas 데이터프레임을 그대로 넣으면 색·디자인을 알아서 처리
- sns가 그래프 본체를 그리고, plt가 제목·축 이름 같은 마무리를 맡는 협업 구조("sns 본체 -> plt 마무리")
- 이 단원의 실데이터: 열처리 공정(`18_열처리.csv`, 297행) - 라인(주간/야간/특근) 세 교대조,
  소입로온도·제어출력·건조출력·세정기·CP값 다섯 숫자 컬럼, 판정(정상/이상) 결과 컬럼

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("../data/18_열처리.csv")
print(df.shape)  # (297, 7)
print(df["라인"].value_counts())  # 주간100, 야간99, 특근98
print(df["판정"].value_counts())  # 정상285, 이상12

## 개념: histplot 기본 사용법 - data와 볼 컬럼 x만 지정하면 분포 완성
- `sns.histplot(data=df, x="컬럼")` - x에는 반드시 숫자형 컬럼만
- 분포를 볼 때는 중심(어디 몰렸나) · 퍼짐(좁게 vs 넓게) · 치우침(한쪽 꼬리) 세 가지를 체크

### 실습 1. 첫 분포 그래프 그리기
- 목표: histplot으로 소입로온도 분포를 그리고 중심·퍼짐 확인
- 단계: ① histplot으로 분포 그리기 -> ② 평균·표준편차로 중심·퍼짐 수치 확인
- 예상 결과: 평균 859.42 근처에 몰린 종 모양 분포, 표준편차 약 2.0

In [ ]:
print(round(df["소입로온도"].mean(), 2), round(df["소입로온도"].std(), 2))  # 859.42 2.0
sns.histplot(data=df, x="소입로온도")
plt.title("소입로온도 분포")
plt.xlabel("소입로온도(℃)")
plt.show()

## 개념: bins·kde - 구간 수 조절과 부드러운 분포 곡선
- bins를 작게 하면 뭉뚱그려지고 크게 하면 자세해짐 - 데이터 개수의 제곱근이 출발점(297개면 약 17)
- kde=True로 막대 위에 분포 곡선을 얹으면 흐름이 더 잘 보이지만, 정확한 개수는 여전히 막대로 확인

**[퀴즈]** 아래 두 줄을 실행하면 봉우리 모양이 어떻게 달라질지 먼저 예상해보기
```python
sns.histplot(data=df, x="소입로온도", bins=5)
sns.histplot(data=df, x="소입로온도", bins=40)
```
정답: bins=5는 막대 5개뿐이라 봉우리 위치만 대략 보이고, bins=40은 297개 데이터가 잘게
쪼개져 군데군데 막대가 비는 듬성듬성한 모양이 됨 - 데이터 개수의 제곱근인 17 근처가 무난

### 실습 2. bins·kde 조절
- 목표: bins 값을 바꿔가며 분포 모양 변화를 관찰하고 kde 곡선 추가
- 단계: ① bins=17(기본 감) -> ② bins=5·40 비교 -> ③ kde=True로 곡선 얹기
- 예상 결과: bins=17에서 봉우리와 퍼짐이 가장 또렷하게 보임

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(data=df, x="소입로온도", bins=5, ax=axes[0])
axes[0].set_title("bins=5")
sns.histplot(data=df, x="소입로온도", bins=17, kde=True, ax=axes[1])
axes[1].set_title("bins=17 + kde")
sns.histplot(data=df, x="소입로온도", bins=40, ax=axes[2])
axes[2].set_title("bins=40")
plt.show()

## 개념: hue로 그룹별 분포 겹쳐 보기
- hue에는 판정·라인처럼 몇 가지로 딱 나뉘는 범주형 컬럼만 - 온도 같은 연속 숫자는 색이 수백 가지가 됨
- 불량(이상) 분포가 정상보다 한쪽으로 치우쳐 있으면 그 센서가 고장 신호일 단서가 됨

### 실습 3. 정상·이상 분포 비교
- 목표: hue로 판정별 소입로온도 분포를 색으로 겹쳐 비교
- 단계: ① x=소입로온도, hue=판정으로 histplot 그리기 -> ② 정상·이상 평균 비교
- 예상 결과: 이상 그룹 평균(856.88)이 정상 그룹(859.53)보다 뚜렷이 낮음 - 온도가 처지는 쪽이 이상 신호

In [ ]:
mean_by_verdict = df.groupby("판정")["소입로온도"].mean().round(2)
print(mean_by_verdict)  # 이상 856.88, 정상 859.53
sns.histplot(data=df, x="소입로온도", hue="판정", kde=True)
plt.title("판정별 소입로온도 분포")
plt.show()

**[강사님께 질문하기]** 이상이 12건뿐이라 정상(285건)에 비해 압도적으로 적은데, 이렇게 개수
차이가 큰 두 그룹을 같은 히스토그램에 겹쳐 그려도 비교가 의미가 있나요?

**답변:** 좋은 지적. 개수(count) 기준 히스토그램은 정상 막대가 훨씬 높게 그려져 이상 쪽
막대는 눈에 잘 안 띄는 착시가 생기기 쉬움. 이런 불균형 상황에서는 sns.histplot에
stat="density"(또는 stat="probability") 옵션을 주면 각 그룹을 "자기 그룹 안에서 차지하는
비율"로 다시 스케일링해서 개수 차이와 무관하게 분포 모양 자체만 공정하게 비교할 수 있음

### 실습 4. 여러 센서 분포 한눈에 보기
- 목표: 반복문으로 소입로온도·제어출력·건조출력 분포를 차례로 그려 비교
- 단계: ① 컬럼 목록을 반복하며 각 분포를 subplot에 그리기 -> ② 제목 달기
- 예상 결과: 세 센서의 분포 모양(중심·퍼짐)을 나란히 확인

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sensor_cols = ["소입로온도", "제어출력", "건조출력"]
for ax, col in zip(axes, sensor_cols):
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(f"{col} 분포")
plt.tight_layout()
plt.show()

## 개념: boxplot이 필요한 이유 - 분포 요약과 이상치 표시를 동시에
- 히스토그램은 분포를 보여줘도 정상 범위를 딱 잘라 말하기 애매함
- 박스플롯 네 부분: 상자(Q1~Q3, 가운데 절반) · 중앙값선 · 수염(정상값의 끝) · 이상치점(수염 밖)
- 이상치 기준은 IQR 방식과 동일: 하한 Q1-1.5xIQR, 상한 Q3+1.5xIQR - boxplot이 자동 계산

### 실습 5. 단일 센서 박스플롯
- 목표: boxplot으로 소입로온도의 사분위수와 이상치를 확인
- 단계: ① Q1·Q3·IQR·경계 직접 계산 -> ② boxplot으로 같은 값이 자동 표시되는지 확인
- 예상 결과: IQR 1.4, 경계 856.7~862.3, 이 범위를 벗어난 이상치 32건(10.77%)

In [ ]:
q1 = df["소입로온도"].quantile(0.25)
q3 = df["소입로온도"].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(round(iqr, 2), round(lower, 2), round(upper, 2))  # 1.4 856.7 862.3
mask = (df["소입로온도"] < lower) | (df["소입로온도"] > upper)
print(mask.sum(), round(mask.mean() * 100, 2))  # 32 10.77
sns.boxplot(data=df, y="소입로온도")
plt.title("소입로온도 박스플롯")
plt.ylabel("소입로온도(℃)")
plt.show()

**[정리]** IQR이 좁으면 이상치 비율이 높아 보이는 게 당연함
- 소입로온도는 대부분 858~860 사이 아주 좁은 범위(IQR 1.4)에 몰려 있어서, 그 밖으로 살짝만
  벗어나도 곧바로 "이상치" 판정을 받음 - 이상치 비율(10.77%)이 높다고 무조건 큰 사고는 아니고,
  원래 변동이 적은 공정이라 작은 흔들림도 민감하게 잡아낸다는 뜻으로 함께 읽어야 함

### 실습 6. 라인별·판정별 박스플롯 비교
- 목표: x=라인, hue=판정으로 교대조별·판정별 소입로온도 분포를 한 번에 비교
- 단계: ① x=라인만으로 라인별 상자 비교 -> ② hue=판정을 더해 이중 비교
- 예상 결과: 세 라인 평균이 859.20~859.64로 큰 차이는 없음 - 라인 자체보다 판정 차이에 주목

In [ ]:
line_mean = df.groupby("라인")["소입로온도"].mean().round(2)
print(line_mean)  # 야간859.64, 주간859.20, 특근859.43
sns.boxplot(data=df, x="라인", y="소입로온도")
plt.title("라인별 소입로온도 분포")
plt.show()

In [ ]:
sns.boxplot(data=df, x="라인", y="소입로온도", hue="판정")
plt.title("라인별 · 판정별 소입로온도 분포")
plt.show()

## 개념: 범주형은 countplot(개수) 또는 barplot(평균)으로 - "A라인이 B라인보다 크다"는 어색한 말
- countplot: x만 지정하면 범주별 개수를 자동 집계해 막대로
- barplot: x(범주)와 y(숫자) 모두 지정하면 범주별 평균을 막대로, 막대 위 세로선은 평균의 신뢰 범위
- 색만 바꿀 땐 hue=x와 같은 컬럼 + legend=False, 다른 기준으로 색을 나눌 땐 hue=다른 컬럼 + 범례 유지

### 실습 7. countplot·barplot 비교
- 목표: 개수는 countplot, 평균은 barplot으로 라인을 비교
- 단계: ① countplot으로 라인별 개수 세기 -> ② barplot으로 라인별 평균 제어출력 비교
- 예상 결과: 개수는 100/99/98로 균등, 평균 제어출력은 라인마다 다름

In [ ]:
line_control = df.groupby("라인")["제어출력"].mean().round(2)
print(line_control)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=df, x="라인", hue="라인", legend=False, ax=axes[0])
axes[0].set_title("라인별 개수")
sns.barplot(data=df, x="라인", y="제어출력", ax=axes[1])
axes[1].set_title("라인별 평균 제어출력")
plt.tight_layout()
plt.show()

### 실습 8. 라인별 정상·이상 비율 비교
- 목표: countplot(hue=판정)과 crosstab으로 라인별 정상·이상 개수를 그래프와 표로 정리
- 단계: ① countplot에 hue=판정 추가 -> ② groupby+unstack(또는 crosstab)으로 표 만들기
- 예상 결과: 주간이 이상 7건으로 세 라인 중 가장 많음(야간3, 특근2)

In [ ]:
cross = pd.crosstab(df["라인"], df["판정"])
print(cross)
sns.countplot(data=df, x="라인", hue="판정")
plt.title("라인별 판정 개수")
plt.show()
# 세 라인의 소입로온도 평균은 거의 같았는데(859.20~859.64) 이상 개수는 주간에 몰려 있음 -
# 온도만으로는 설명이 안 되는 다른 원인(제어출력·세정기 등)을 함께 봐야 한다는 신호

# Seaborn 기초 - 관계분석과 시각화리포트

## 문제 상황
단변량에서 이변량으로 - 두 변수가 함께 어떻게 움직이는가
- 온도가 오르면 다른 값도 같이 오르는가라는 질문은 단변량 분포보다 한 단계 더 중요한 질문
- 관계를 알면 한 센서로 다른 센서를 추측하고, 평소 관계가 깨지는 순간을 이상 신호로 포착 가능

## 개념: scatterplot 기본 사용법 - x·y 모두 숫자형 컬럼 지정
- `sns.scatterplot(data=df, x="컬럼1", y="컬럼2")` - histplot은 x 하나였지만 scatterplot은 두 축 모두 필요
- 점이 겹치면 alpha로 반투명, 범주형 그룹 구분은 x·y가 아닌 hue 자리에

### 실습 1. 산점도로 두 변수 관계 보기
- 목표: 소입로온도-제어출력의 관계를 산점도로 확인
- 단계: ① scatterplot으로 점 찍기 -> ② corr()로 상관계수 확인
- 예상 결과: 온도가 높을수록 제어출력이 낮아지는 뚜렷한 음의 관계(-0.736)

In [ ]:
corr_tc = df["소입로온도"].corr(df["제어출력"])
print(round(corr_tc, 3))  # -0.736
sns.scatterplot(data=df, x="소입로온도", y="제어출력", alpha=0.5)
plt.title("소입로온도-제어출력 관계")
plt.show()

**[정리]** 왜 온도가 높을수록 제어출력이 낮아지는가 - 상관은 인과가 아니지만 그럴듯한 설명은 가능
- 소입로가 이미 목표 온도 근처(고온)에 도달했다면 히터를 약하게만 돌려도 온도가 유지되지만,
  온도가 낮으면 목표치까지 끌어올리려고 제어출력을 강하게 씀 - "온도가 출력을 낮췄다"보다는
  "제어 시스템이 목표 온도에 맞춰 출력을 조절한 결과"로 읽는 것이 더 정확한 해석

### 실습 2. 판정별 산점도 패턴
- 목표: hue=판정으로 이상 포인트가 어느 영역에 몰리는지 확인
- 단계: ① x=소입로온도, y=제어출력, hue=판정으로 산점도 그리기 -> ② 이상 포인트의 위치 파악
- 예상 결과: 이상(12건)이 특정 온도·출력 구간에 몰려 있는지 확인

In [ ]:
print(df.groupby("판정")[["소입로온도", "제어출력"]].mean().round(2))
sns.scatterplot(data=df, x="소입로온도", y="제어출력", hue="판정", alpha=0.7)
plt.title("판정별 소입로온도-제어출력 관계")
plt.show()

## 개념: 상관계수와 상관행렬 - -1~+1 사이 숫자 하나로 관계의 방향·강도를 요약
- 부호(+/-)는 방향, 크기(절댓값)는 강도 - 0.7 이상 강함 / 0.4~0.7 중간 / 0.2~0.4 약함
- `df.corr(numeric_only=True)`로 숫자 컬럼 쌍 전체의 상관계수를 한 번에 계산(상관행렬)
- 대각선은 항상 1.00(자기 자신), 대각선 기준 위아래는 대칭이라 절반만 봐도 모든 정보 확보

**[퀴즈]** 상관행렬에서 대각선 값은 항상 얼마일지, 그리고 왜 그런지 먼저 예상해보기

정답: 항상 1.00
(같은 컬럼을 자기 자신과 비교하면 완벽하게 같이 움직이므로 상관계수가 최댓값인 1이 됨 -
대각선이 1이 아니면 오히려 계산이 잘못됐다는 신호)

### 실습 3. 상관관계 heatmap
- 목표: 다섯 숫자 컬럼의 상관행렬을 heatmap으로 그리고 강한 관계 쌍을 찾기
- 단계: ① corr()로 상관행렬 만들기 -> ② heatmap(annot, fmt, cmap, center)으로 표현
- 예상 결과: 소입로온도-제어출력이 -0.74로 가장 강한 관계, 나머지 쌍은 대부분 0에 가까움

In [ ]:
corr = df.corr(numeric_only=True)
print(corr.round(2))
plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("열처리 공정 상관관계")
plt.show()

**[강사님께 질문하기]** heatmap에 원본 df를 그대로 넣지 않고 꼭 df.corr() 결과를 먼저 만들어서
넣어야 하는 이유가 있나요?

**답변:** heatmap은 "행 x 열 칸마다 숫자 하나를 색으로 칠하는" 범용 도구일 뿐, 그 숫자가
상관계수여야 한다는 규칙은 없음. 원본 df를 그대로 넣으면 df의 각 행(개별 측정 샘플)이
칸이 되어 버려 "컬럼끼리의 관계"라는 의도와 전혀 다른 그림이 그려짐. 반드시 corr()로
"컬럼 x 컬럼" 모양의 표를 먼저 만들어야 heatmap이 상관행렬을 그리는 데 쓰일 수 있음

## 개념: 그래프 선택 치트시트와 리포트 구성 순서
- 분포(숫자1개)->histplot/boxplot, 범주별 개수->countplot, 두 변수 관계->scatterplot,
  범주별 평균->barplot, 여러 변수 관계->heatmap
- EDA(탐색)로 발견한 것 중 메시지가 분명한 그래프만 골라 전체->분포->비교->관계 순으로 리포트 구성
- 그래프가 실제로 보여준 것만 사실로 적기 - "상관계수가 높다"고 "원인이다"라고 단정하지 않기

### 실습 4. 종합 리포트 - 분포·박스플롯
- 목표: subplots로 라인별 개수·소입로온도 분포·박스플롯을 한 화면에 배치
- 단계: ① subplots로 가로 3칸 만들기 -> ② 각 칸에 개수·분포·박스플롯 그리기 -> ③ tight_layout
- 예상 결과: 개수·분포·박스플롯이 나란한 3칸 리포트

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(data=df, x="라인", hue="라인", legend=False, ax=axes[0])
axes[0].set_title("라인별 개수")
sns.histplot(data=df, x="소입로온도", kde=True, ax=axes[1])
axes[1].set_title("소입로온도 분포")
sns.boxplot(data=df, x="라인", y="소입로온도", ax=axes[2])
axes[2].set_title("라인별 소입로온도 박스플롯")
plt.tight_layout()
plt.show()

### 실습 5. 종합 리포트 - 산점도·heatmap
- 목표: subplots로 판정별 산점도와 상관 heatmap을 한 화면에 배치
- 단계: ① subplots로 가로 2칸 만들기 -> ② 왼쪽 산점도, 오른쪽 heatmap -> ③ tight_layout
- 예상 결과: 판정별 산점도와 상관 heatmap이 나란한 리포트

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.scatterplot(data=df, x="소입로온도", y="제어출력", hue="판정", alpha=0.7, ax=axes[0])
axes[0].set_title("판정별 온도-제어출력 관계")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("상관관계")
plt.tight_layout()
plt.show()

### 실습 6. 시각화 리포트 통합·저장
- 목표: 핵심 그래프 네 개를 2x2로 모아 리포트 파일로 저장(show와 savefig 둘 다)
- 단계: ① subplots로 2x2 격자 만들기 -> ② 분포·박스플롯·산점도·heatmap을 네 칸에 배치
  -> ③ show로 확인, savefig로 저장하고 파일 존재 확인
- 예상 결과: 분포·박스·관계·상관을 담은 2x2 리포트가 화면에 표시되고 파일로도 저장됨

In [ ]:
import os

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
sns.histplot(data=df, x="소입로온도", kde=True, ax=axes[0, 0])
axes[0, 0].set_title("소입로온도 분포")
sns.boxplot(data=df, x="라인", y="소입로온도", ax=axes[0, 1])
axes[0, 1].set_title("라인별 박스플롯")
sns.scatterplot(data=df, x="소입로온도", y="제어출력", hue="판정", alpha=0.7, ax=axes[1, 0])
axes[1, 0].set_title("온도-제어출력 관계")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1, 1])
axes[1, 1].set_title("상관관계")
plt.suptitle("열처리 공정 시각화 리포트")
plt.tight_layout()
report_path = os.path.join("..", "data", "_tmp_sns_final_report.png")
plt.savefig(report_path, dpi=100)
plt.show()
print(os.path.exists(report_path))  # True
os.remove(report_path)

**[정리]** 사실 -> 의미 -> 행동
- 사실: 소입로온도-제어출력 상관계수 -0.74(강한 음의 관계), 판정별 이상 12건은 주간(7)에 가장 많음
- 의미: 온도-출력 관계는 제어 시스템이 정상 작동한다는 신호이고, 이상은 라인 온도 차이보다는
  주간 특유의 다른 조건(작업 빈도·세정 주기 등)과 관련됐을 가능성이 큼
- 행동: 주간 라인의 이상 발생 시점에서 세정기·CP값을 함께 점검해 실제 원인을 좁혀볼 것을 제안